In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Merging all three into one dataset

housing = pd.read_csv(r"C:\Users\prith\Downloads\airbnb-data-pipeline\airbnb-data-pipeline\data\processed\listing_clean.csv")
reviews = pd.read_csv(r"C:\Users\prith\Downloads\airbnb-data-pipeline\airbnb-data-pipeline\data\processed\reviews_clean.csv")
calendar = pd.read_csv(r"C:\Users\prith\Downloads\airbnb-data-pipeline\airbnb-data-pipeline\data\processed\calendar_clean.csv")

housing = housing.merge(reviews, left_on = 'id', right_on = 'listing_id', how='left')
housing = housing.merge(calendar, left_on = 'id', right_on = 'listing_id', how='left')

In [ ]:
print(housing.head(10).to_string())

In [ ]:
print(housing.dtypes.to_string())

In [ ]:
print(housing.shape)

In [ ]:
print((housing.isnull().mean()*100).to_string())

In [ ]:
#filling the null values in few_columns
housing['review_count'] = housing['review_count'].fillna(0)
housing['avg_comment_length'] = housing['avg_comment_length'].fillna(0)

housing = housing.drop(columns=['listing_id_x', 'listing_id_y']) #duplicate listing_id

# droping the date columns from reviews as we already have days_since in listings
housing = housing.drop(columns=['first_review_date', 'last_review_date'])


In [ ]:
print((housing.isnull().mean()*100).to_string())

In [ ]:
housing.shape

In [ ]:
print(housing.head(10).to_string())

In [ ]:
#doing the feature engineering (creating new attributes):

housing['price_per_person'] = housing['price']/housing['accommodates']

housing['price_per_bedroom'] = housing['price']/housing['bedrooms'].replace(0,1)

housing['log_price'] = np.log1p(housing['price'])

housing['host_experience_level'] = pd.cut(
    housing['hosts_time_as_host_years'],
    bins= [-1, 1, 3, 9999],
    labels= ['New', 'Intermediate', 'Experienced']
)

housing['avail_0_30'] = housing['availability_30'] / 30
housing['avail_31_60'] = (housing['availability_60'] - housing['availability_30']) / 30
housing['avail_61_90'] = (housing['availability_90'] - housing['availability_60']) / 30
housing['avail_91_365'] = (housing['availability_365'] - housing['availability_90']) / 275

housing['total_availability_score'] = (
    housing['avail_0_30'] * 0.4 +
    housing['avail_31_60'] * 0.3 +
    housing['avail_61_90'] * 0.2 +
    housing['avail_91_365'] * 0.1
)


In [ ]:
housing[['price_per_person', 'price_per_bedroom', 'log_price', 'total_availability_score']].describe()

In [ ]:
#adding a few more new attributes
review_cols = ['review_scores_rating', 'review_scores_accuracy', 
               'review_scores_cleanliness', 'review_scores_checkin',
               'review_scores_communication', 'review_scores_location', 
               'review_scores_value']

housing['review_score_avg'] = housing[review_cols].mean(axis=1)

median_reviews = housing['number_of_reviews'].median()
housing['is_high_demand'] = (housing['number_of_reviews'] > median_reviews).astype(int)

In [ ]:
housing.shape

In [ ]:
print(housing.head(10).to_string())

In [ ]:
#dropping few columns that are not much needed
cols_to_drop = [
    'hosts_time_as_user_years',
    'hosts_time_as_user_months',
    'hosts_time_as_host_months',

    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'avail_0_30',
    'avail_31_60',
    'avail_61_90',
    'avail_91_365',
    'availability_eoy',
    'total_days',           # every row is 365, zero variance

    'number_of_reviews_l30d',
    'number_of_reviews_ly',
    'review_count',         

    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_value',

    'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms',
    'host_listings_count',  

    'avg_comment_length',   # weak signal, review_score_avg captures quality better
    'reviews_per_month',    
]

housing = housing.drop(columns=cols_to_drop)

In [ ]:
print(housing.shape)
print(housing.columns.tolist())

In [ ]:
#removing the id and host_id
housing = housing.drop(columns=['id', 'host_id'])
print(housing.shape)

In [ ]:
print(housing.head(10).to_string())

In [ ]:
print(housing.dtypes.to_string())

In [ ]:
print(housing.head(10).to_string())

In [ ]:
print(housing.describe().to_string())

In [ ]:
#as we can see that in the price, price_per_person, price_per_bedroom and minimum and maximum _nights
#has extreme outlier, as the mean and median are low but the max is very high
#therefore capping is required.
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(f"{col}: capped at [{lower:.2f}, {upper:.2f}]")
    df[col] = df[col].clip(lower, upper)
    return df

# drop low variance columns first
housing = housing.drop(columns=['minimum_nights', 'minimum_nights_avg_ntm'])

# cap outliers
for col in ['price', 'price_per_person', 'price_per_bedroom',
            'maximum_nights', 'maximum_nights_avg_ntm',
            'number_of_reviews', 'number_of_reviews_ltm',
            'calculated_host_listings_count']:
    housing = cap_outliers(housing, col)

# verify
print("\nShape:", housing.shape)
print("Nulls:", housing.isnull().sum().sum())

In [ ]:
print(housing.describe().to_string())

In [ ]:
print(housing.head(1000).to_string())

In [ ]:
housing['availability_rate'] = housing['availability_rate'] / 365

In [ ]:
# drop is_high_demand
housing = housing.drop(columns=['is_high_demand'])

#as the model would learn if "high no of reviews = is high demand"

In [ ]:
print(housing['number_of_reviews_ltm'].value_counts().head(10))


In [ ]:
print(housing.isnull().sum()[housing.isnull().sum() > 0])

In [ ]:
#as we can see that capping created many Nan values so make some changes, like
#in host_exp level we can change the bin from 0 <-> -1
# and drop no_of_reviews_ltmm  as so many rows are 0 therefore now its not that important
housing = housing.drop(columns=['number_of_reviews_ltm'])
#i have updated the values in the host exp level so it might not appear in the prev cell now.

In [ ]:
print("Nulls:", housing.isnull().sum().sum())
print("Shape:", housing.shape)

In [ ]:
print(housing.duplicated().to_string())
#duplicates are created as the capped values gives NaN. Therefore we have to remove them.

In [ ]:
print(housing.shape)
housing = housing.drop_duplicates()
print(housing.shape)

In [ ]:
print(housing.head(10).to_string())

In [ ]:
print(housing.columns.tolist())
print(housing.shape)
print(housing.isnull().sum().sum())

In [ ]:
housing.to_csv(r'C:\Users\prith\Downloads\airbnb-data-pipeline\airbnb-data-pipeline\data\processed\featured_engineered_data.csv', index=False)